# Assignment 2: Data Cleaning

Group 33: Michael Massaad (300293612) & Gabriel Zohrob (300309391)

Work Split: 
- Michael: Implemented the Part 1: Validity Checker
- Gabriel: Implemented the Part 2: Missing Data and Imputation

## Part 1: Validity Checker

### Description of the World Happiness Report (2019) dataset

Link: https://www.kaggle.com/datasets/unsdsn/world-happiness?resource=download&select=2019.csv

Author: Sustainable Development Solutions Network

Purpose: The World Happiness Report dataset was created to measure and analyze the happiness levels of countries worldwide. The report ranks countries based on survey responses that evaluate overall life satisfaction, along with several contributing socio-economic factors such as GDP per capita, social support, life expectancy, freedom, generosity, and perceptions of corruption.

The dataset is based on real-world survey and economic data and is not synthetic.

Shape: 156 rows and 9 columns, Each row represents one country and its associated happiness metrics.

Features:
1. **Overall rank** (Numerical – Integer): The ranking position of the country based on its happiness score (1 = happiest country).

2. **Country or region** (Categorical – String): The name of the country being evaluated.

3. **Score** (Numerical – Float): The overall happiness score assigned to the country.

4. **GDP per capita** (Numerical – Float): A measure of economic output per person, contributing to the happiness score.

5. **Social support** (Numerical – Float): A measure of perceived social support within the country.

6. **Healthy life expectancy** (Numerical – Float): The average number of years a person is expected to live in good health.

7. **Freedom to make life choices** (Numerical – Float): A measure of the population’s perceived freedom in making life decisions.

8. **Generosity** (Numerical – Float): A measure of charitable behavior and generosity within the population.

9. **Perceptions of corruption** (Numerical – Float): A measure of perceived corruption in government and business institutions.

### Loading Dataset:

In [339]:
import pandas as pd

DATA_URL = "https://github.com/michaelmassaad02/CSI4142_Assignment2/raw/main/2019.csv"

data_original = pd.read_csv(DATA_URL)

print("Shape: ", data_original.shape)
print("Columns: \n", data_original.columns.tolist())

data_original.head()


Shape:  (156, 9)
Columns: 
 ['Overall rank', 'Country or region', 'Score', 'GDP per capita', 'Social support', 'Healthy life expectancy', 'Freedom to make life choices', 'Generosity', 'Perceptions of corruption']


,Overall rank,Country or region,Score,GDP per capita,Social support,Healthy life expectancy,Freedom to make life choices,Generosity,Perceptions of corruption
0,1,Finland,7.769,1.340,1.587,0.986,0.596,0.153,0.393
1,2,Denmark,7.600,1.383,1.573,0.996,0.592,0.252,0.410
2,3,Norway,7.554,1.488,1.582,1.028,0.603,0.271,0.341
3,4,Iceland,7.494,1.380,1.624,1.026,0.591,0.354,0.118
4,5,Netherlands,7.488,1.396,1.522,0.999,0.557,0.322,0.298


### Data Type Error Check:

In this test, we will verify the data type of a numerical value. A data type error occurs when a value does not match the expected data type of a column. 

In [340]:
# Introducing around 5% error by inserting the string "invalid" into the numerical column
import numpy as np

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
data_copy = data_original.copy()

def sample_indices(data, frac):
    n = len(data)
    k = max(1, int(round(n * frac)))
    return rng.choice(data.index.to_numpy(), size=k, replace=False)

error_rows = sample_indices(data_copy, 0.05)
data_copy.loc[error_rows, "GDP per capita"] = "invalid"


C:\Users\mike2\AppData\Local\Temp\ipykernel_532\2869440817.py:14: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'invalid' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  data_copy.loc[error_rows, "GDP per capita"] = "invalid"


In [341]:
# Error Finding: Identifying rows with non-numeric values in the "GDP per capita" column
def find_non_numeric_rows(data, column):
    non_numeric_rows = data[~data[column].apply(lambda x: isinstance(x, (int, float)))]
    return non_numeric_rows

errors = find_non_numeric_rows(data_copy, "GDP per capita")
print("Number of datatype errors found: ", len(errors))

errors.head(8)


Number of datatype errors found:  8


,Overall rank,Country or region,Score,GDP per capita,Social support,Healthy life expectancy,Freedom to make life choices,Generosity,Perceptions of corruption
13,14,Luxembourg,7.090,invalid,1.479,1.012,0.526,0.194,0.316
66,67,Pakistan,5.653,invalid,0.886,0.535,0.313,0.220,0.098
98,99,Ivory Coast,4.944,invalid,0.808,0.232,0.352,0.154,0.090
108,109,Cambodia,4.700,invalid,1.122,0.637,0.609,0.232,0.062
116,117,Iran,4.548,invalid,0.842,0.785,0.305,0.270,0.125
132,133,Ukraine,4.332,invalid,1.390,0.739,0.178,0.187,0.010
152,153,Tanzania,3.231,invalid,0.885,0.499,0.417,0.276,0.147
154,155,Central African Republic,3.083,invalid,0.000,0.105,0.225,0.235,0.035


In [342]:
# Qualitative results + examples

error_indices = errors.index

examples = pd.DataFrame({
    "Original GDP per capita": data_original.loc[error_indices, "GDP per capita"].astype(str),
    "Invalid GDP per capita": data_copy.loc[error_indices, "GDP per capita"].astype(str)
})    

examples.head(8)


# After intentionally introducing datatype errors by inserting the string value `"invalid"` into approximately 5% of the rows in the **"GDP per capita"** column, the validity checker successfully detected all non-numeric entries.
#The checker works by verifying whether each value in the column is of type `int` or `float`. Since GDP per capita is a numerical attribute, any string or other value represents invalid data. The results confirm that the modified rows were correctly identified as datatype errors.
#These errors could negatively impact statistical calculations such as computing the mean GDP per capita or performing correlation analysis. Detecting datatype errors ensures that numerical operations remain valid and reliable.



,Original GDP per capita,Invalid GDP per capita
13,1.609,invalid
66,0.677,invalid
98,0.569,invalid
108,0.574,invalid
116,1.1,invalid
132,0.82,invalid
152,0.476,invalid
154,0.026,invalid


### Range Error Check
In this test, we will verify the range of a numerical value. A range error occurs when a value falls outside the expected minimum or maximum bounds of a column.

For this dataset, the "Score" column represents the happiness score of each country. Happiness scores are expected to fall within the range of 0 to 10. Any value below 0 or above 10 would indicate invalid data.

This test will identify rows where the "Score" value is outside the valid range [0, 10].

In [343]:
# Introducing around 5% error by adding negative values and values greater than 10 to the score column

data_copy = data_original.copy()

range_error_rows = sample_indices(data_copy, 0.05)

half = len(range_error_rows) // 2 # Split the indices into two halves, one for negative values and one for values greater than 10

data_copy.loc[range_error_rows[:half], "Score"] = -1  # Assign negative values to the first half

data_copy.loc[range_error_rows[half:], "Score"] = 11  # Assign values greater than 10 to the second half


In [344]:
# Error Finding: Identifying rows with out-of-range values in the "Score" column
def find_out_of_range_rows(data, column, min_value, max_value):
    out_of_range_rows = data[(data[column] < min_value) | (data[column] > max_value)]
    return out_of_range_rows

errors = find_out_of_range_rows(data_copy, "Score", 0, 10)
print("Number of range errors found: ", len(errors))

errors.head(8)

Number of range errors found:  8


,Overall rank,Country or region,Score,GDP per capita,Social support,Healthy life expectancy,Freedom to make life choices,Generosity,Perceptions of corruption
19,20,Czech Republic,11.0,1.269,1.487,0.920,0.457,0.046,0.036
28,29,Qatar,11.0,1.684,1.313,0.871,0.555,0.220,0.167
57,58,Japan,-1.0,1.327,1.419,1.088,0.445,0.069,0.140
68,69,Philippines,-1.0,0.807,1.293,0.657,0.558,0.117,0.107
76,77,Dominican Republic,-1.0,1.015,1.401,0.779,0.497,0.113,0.101
77,78,Bosnia and Herzegovina,11.0,0.945,1.212,0.845,0.212,0.263,0.006
117,118,Guinea,-1.0,0.380,0.829,0.375,0.332,0.207,0.086
127,128,Mali,11.0,0.385,1.105,0.308,0.327,0.153,0.052


In [345]:
# Qualitative results + examples

error_indices = errors.index

examples = pd.DataFrame({
    "Original Score": data_original.loc[error_indices, "Score"].astype(str),
    "Invalid Score": data_copy.loc[error_indices, "Score"].astype(str)
})    

examples.head(8)


# After intentionally introducing range errors by assigning values outside the valid range (0–10) to approximately 5% of the rows in the **"Score"** column, the validity checker successfully detected all out-of-range entries.
# The checker works by verifying whether each value falls within the expected interval [0, 10]. Any value below 0 or above 10 is considered invalid and is flagged as a range error.
# Range errors can distort statistical measures such as averages and rankings, potentially leading to incorrect interpretations of country happiness levels. Detecting these errors ensures the dataset remains logically consistent and suitable for analysis.


,Original Score,Invalid Score
19,6.852,11.0
28,6.374,11.0
57,5.886,-1.0
68,5.631,-1.0
76,5.425,-1.0
77,5.386,11.0
117,4.534,-1.0
127,4.39,11.0


### Format Error Check:

In this test, we verify the format of a numerical value. A format error occurs when a value does not follow the expected structural pattern.

For this dataset, the "Perceptions of corruption" column contains decimal values that are expected to have exactly three decimal places.

This test will identify rows where the "Perceptions of corruption" value does not follow the required three-decimal format.

In [346]:
# Introducing around 5% error by inserting the number of "0.1" and "0.12345" into the numerical column

# Convert to all original values as strings with 3 decimals
data_originalCopy = data_original.copy() 

data_originalCopy["Perceptions of corruption"] = \
    data_originalCopy["Perceptions of corruption"].map(lambda x: f"{x:.3f}")

data_copy = data_originalCopy.copy()

format_error_indices = sample_indices(data_copy, frac=0.05)

# Some with 1 decimal, some with 5 decimals
half = len(format_error_indices) // 2

# Introduce invalid formatted numbers
data_copy.loc[format_error_indices[:half], "Perceptions of corruption"] = str(0.1)
data_copy.loc[format_error_indices[half:], "Perceptions of corruption"] = str(0.12345)


In [347]:
# Error Finding: exactly 3 decimal places required
def find_format_errors(data, column):
    pattern = r"^\d+\.\d{3}$"
    return data[~data[column].astype(str).str.match(pattern)]

errors = find_format_errors(data_copy, "Perceptions of corruption")

print("Number of format errors found:", len(errors))

errors.head(8)

Number of format errors found: 8


,Overall rank,Country or region,Score,GDP per capita,Social support,Healthy life expectancy,Freedom to make life choices,Generosity,Perceptions of corruption
9,10,Austria,7.246,1.376,1.475,1.016,0.532,0.244,0.1
13,14,Luxembourg,7.090,1.609,1.479,1.012,0.526,0.194,0.12345
34,35,El Salvador,6.253,0.794,1.242,0.789,0.430,0.093,0.1
67,68,Russia,5.648,1.183,1.452,0.726,0.334,0.082,0.12345
84,85,Nigeria,5.265,0.696,1.111,0.245,0.426,0.215,0.12345
129,130,Sri Lanka,4.366,0.949,1.265,0.831,0.470,0.244,0.1
133,134,Ethiopia,4.286,0.336,1.033,0.532,0.344,0.209,0.1
135,136,Uganda,4.189,0.332,1.069,0.443,0.356,0.252,0.12345


In [348]:
# Qualitative results + examples

error_indices = errors.index

examples = pd.DataFrame({
    "Original Perceptions of corruption": data_originalCopy.loc[error_indices, "Perceptions of corruption"].astype(str),
    "Invalid formatted value": data_copy.loc[error_indices, "Perceptions of corruption"].astype(str)
})   

examples.head(8)

# After intentionally introducing format errors into approximately 5% of the rows in the "Perceptions of corruption" column, the validity checker successfully detected values that do not follow the required format of **exactly three digits after the decimal point**.
# Because the original dataset stores this column as floating-point numbers (which do not preserve trailing zeros), we standardized the column into a string representation with three decimals before injecting errors. Then we introduced invalid formats such as values with too few decimal places (e.g., "0.1") or too many (e.g., "0.12345"). The checker uses a regular expression to confirm the pattern **digits + '.' + exactly 3 digits**.
# Format errors like these can create inconsistencies in reporting, exporting, and downstream validation, especially when a dataset is expected to follow a strict numeric formatting standard.

,Original Perceptions of corruption,Invalid formatted value
9,0.226,0.1
13,0.316,0.12345
34,0.074,0.1
67,0.031,0.12345
84,0.041,0.12345
129,0.047,0.1
133,0.100,0.1
135,0.060,0.12345


### Consistency Error Check:

In this test, we will verify consistency between two related attributes. A consistency error occurs when values across columns contradict an expected relationship.

For this dataset, the "Overall rank" and "Score" columns are related: a better rank (closer to 1) should generally correspond to a higher happiness score. Therefore, we expect the "Score" values to decrease as the rank number increases.

This test will identify rows that violate this expected ranking-score consistency.

In [349]:
data_copy = data_original.copy()

# Select ~5% of rows to be part of inconsistencies
k = max(1, int(round(0.05 * len(data_copy))))

# Take k/2 from best ranks and k/2 from worst ranks
top_k = k // 2
bottom_k = k - top_k

top_indices = data_copy.nsmallest(top_k, "Overall rank").index
bottom_indices = data_copy.nlargest(bottom_k, "Overall rank").index

# Swap their Score values to create inconsistency
temp_scores = data_copy.loc[top_indices, "Score"].copy()
data_copy.loc[top_indices, "Score"] = data_copy.loc[bottom_indices, "Score"].values
data_copy.loc[bottom_indices, "Score"] = temp_scores.values

In [350]:
# Error Finding: Identify rank-score inconsistencies
def find_rank_score_inconsistencies(data, rank_col="Overall rank", score_col="Score"):
    sorted_data = data.sort_values(rank_col).copy()
    # Score should generally decrease as rank increases
    increasing = sorted_data[score_col].diff() > 0
    return sorted_data[increasing]

errors = find_rank_score_inconsistencies(data_copy, "Overall rank", "Score")
print("Number of consistency errors found:", len(errors))

errors.head(8)

Number of consistency errors found: 8


,Overall rank,Country or region,Score,GDP per capita,Social support,Healthy life expectancy,Freedom to make life choices,Generosity,Perceptions of corruption
1,2,Denmark,3.083,1.383,1.573,0.996,0.592,0.252,0.410
2,3,Norway,3.203,1.488,1.582,1.028,0.603,0.271,0.341
3,4,Iceland,3.231,1.380,1.624,1.026,0.591,0.354,0.118
4,5,Netherlands,7.488,1.396,1.522,0.999,0.557,0.322,0.298
152,153,Tanzania,7.494,0.476,0.885,0.499,0.417,0.276,0.147
153,154,Afghanistan,7.554,0.350,0.517,0.361,0.000,0.158,0.025
154,155,Central African Republic,7.600,0.026,0.000,0.105,0.225,0.235,0.035
155,156,South Sudan,7.769,0.306,0.575,0.295,0.010,0.202,0.091


In [351]:
# Qualitative results + examples

error_indices = errors.index

examples = pd.DataFrame({
    "Overall rank": data_copy.loc[error_indices, "Overall rank"].astype(str),
    "Country or region": data_copy.loc[error_indices, "Country or region"].astype(str),
    "Original Score": data_originalCopy.loc[error_indices, "Score"].astype(str),
    "Inconsistent Score": data_copy.loc[error_indices, "Score"].astype(str)
})

examples.head(8)

# After intentionally introducing consistency errors by swapping "Score" values between some top-ranked and bottom-ranked countries, the validity checker successfully detected violations of the expected relationship between "Overall rank" and "Score".
# Since rank 1 represents the best-performing country, scores should generally decrease as rank values increase. The checker sorts the dataset by rank and flags positions where the score increases compared to the previous rank, which indicates an inconsistency.
# Consistency errors can lead to misleading interpretations (e.g., a low-ranked country appearing to have an unusually high score). Detecting these issues helps ensure the dataset preserves logical relationships between related attributes.


,Overall rank,Country or region,Original Score,Inconsistent Score
1,2,Denmark,7.6,3.083
2,3,Norway,7.554,3.203
3,4,Iceland,7.494,3.231
4,5,Netherlands,7.488,7.488
152,153,Tanzania,3.231,7.494
153,154,Afghanistan,3.203,7.554
154,155,Central African Republic,3.083,7.6
155,156,South Sudan,2.853,7.769


### Uniqueness Error Check:

In this test, we will verify uniqueness of an attribute. A uniqueness error occurs when a value that is expected to be unique appears more than once in a dataset.

For this dataset, the "Overall rank" column should be unique because each rank value should correspond to exactly one country.

This test will identify duplicate values in the "Overall rank" column.

In [352]:
# Introducing Uniqueness errors for 5% of the rows
data_copy = data_original.copy()

# Select ~5% of rows
dup_indices = sample_indices(data_copy, frac=0.05)

# Force duplicates by copying the rank from the first selected row
target_rank = data_copy.loc[dup_indices[0], "Overall rank"]
data_copy.loc[dup_indices, "Overall rank"] = target_rank


In [353]:
# Error Finding: Identifying duplicate values in the "Overall rank" column
def find_duplicate_rows(data, column):
    return data[data.duplicated(subset=[column], keep=False)]

errors = find_duplicate_rows(data_copy, "Overall rank")
print("Number of uniqueness errors found:", len(errors))

errors.head(8)

Number of uniqueness errors found: 8


,Overall rank,Country or region,Score,GDP per capita,Social support,Healthy life expectancy,Freedom to make life choices,Generosity,Perceptions of corruption
30,31,Panama,6.321,1.149,1.442,0.910,0.516,0.109,0.054
56,31,Mauritius,5.888,1.120,1.402,0.798,0.498,0.215,0.060
66,31,Pakistan,5.653,0.677,0.886,0.535,0.313,0.220,0.098
103,31,Gabon,4.799,1.057,1.183,0.571,0.295,0.043,0.055
117,31,Guinea,4.534,0.380,0.829,0.375,0.332,0.207,0.086
119,31,Gambia,4.516,0.308,0.939,0.428,0.382,0.269,0.167
134,31,Swaziland,4.212,0.811,1.149,0.000,0.313,0.074,0.135
144,31,Burundi,3.775,0.046,0.447,0.380,0.220,0.176,0.180


In [354]:
# Qualitative results + examples

error_indices = errors.index

examples = pd.DataFrame({
    "Original rank": data_original.loc[error_indices, "Overall rank"].astype(str),
    "Duplicated rank": data_copy.loc[error_indices, "Overall rank"].astype(str),
    "Country or region": data_copy.loc[error_indices, "Country or region"].astype(str)
})

examples.head(8)

# After intentionally introducing uniqueness errors by assigning the same "Overall rank" value to approximately 5% of the rows, the validity checker successfully detected duplicated ranks.
# The checker identifies duplicates by searching for repeated values in a column that is expected to be unique. Since each rank should correspond to exactly one country, any repeated rank value represents a uniqueness error.
# Uniqueness errors can cause ambiguity and incorrect joins/merges, especially when rank is used as a key. Detecting them helps preserve the integrity of identifiers in the dataset.

,Original rank,Duplicated rank,Country or region
30,31,31,Panama
56,57,31,Mauritius
66,67,31,Pakistan
103,104,31,Gabon
117,118,31,Guinea
119,120,31,Gambia
134,135,31,Swaziland
144,145,31,Burundi


### Presence Error Check:

In this test, we verify the presence of required values. A presence error occurs when a value that is expected to exist is missing (e.g., NaN or null).

For this dataset, we assume that the "Social support" column should contain a value for every country. Missing values in this column would represent incomplete data.

This test will identify rows where the "Social support" value is missing.

In [367]:
data_copy = data_original.copy()

# Select ~5% of rows
presence_error_indices = sample_indices(data_copy, frac=0.05)

# Introduce missing values
data_copy.loc[presence_error_indices, "Social support"] = np.nan

In [368]:
# Error Finding: Identifying missing values in "Social support"
def find_missing_values(data, column):
    return data[data[column].isna()]

errors = find_missing_values(data_copy, "Social support")

print("Number of presence errors found:", len(errors))
errors.head(8)

Number of presence errors found: 8


,Overall rank,Country or region,Score,GDP per capita,Social support,Healthy life expectancy,Freedom to make life choices,Generosity,Perceptions of corruption
0,1,Finland,7.769,1.340,NaN,0.986,0.596,0.153,0.393
7,8,New Zealand,7.307,1.303,NaN,1.026,0.585,0.330,0.380
34,35,El Salvador,6.253,0.794,NaN,0.789,0.430,0.093,0.074
45,46,Kosovo,6.100,0.882,NaN,0.758,0.489,0.262,0.006
102,103,Congo (Brazzaville),4.812,0.673,NaN,0.508,0.372,0.105,0.093
109,110,Palestinian Territories,4.696,0.657,NaN,0.672,0.225,0.103,0.066
131,132,Chad,4.350,0.350,NaN,0.192,0.174,0.198,0.078
135,136,Uganda,4.189,0.332,NaN,0.443,0.356,0.252,0.060


In [369]:
# Qualitative results + examples

error_indices = errors.index

examples = pd.DataFrame({
    "Country or region": data_copy.loc[error_indices, "Country or region"].astype(str),
    "Original Social support": data_original.loc[error_indices, "Social support"].astype(str),
    "Missing Social support": data_copy.loc[error_indices, "Social support"].astype(str)
})

examples.head(8)

,Country or region,Original Social support,Missing Social support
0,Finland,1.587,nan
7,New Zealand,1.557,nan
34,El Salvador,1.242,nan
45,Kosovo,1.232,nan
102,Congo (Brazzaville),0.799,nan
109,Palestinian Territories,1.247,nan
131,Chad,0.766,nan
135,Uganda,1.069,nan


### Length Error Check:

In this test, we verify the length of textual values. A length error occurs when a value exceeds or falls below an expected number of characters.

For this dataset, the "Country or region" column should contain realistic country names. Country names are expected to have a reasonable length between 4 and 50 characters.

This test will identify rows where the length of the "Country or region" value falls outside the valid range [4, 50].

In [370]:
data_copy = data_original.copy()

# Select ~5% of rows
length_error_indices = sample_indices(data_copy, frac=0.05)

half = len(length_error_indices) // 2

# Introduce short invalid country names (< 4 characters)
data_copy.loc[length_error_indices[:half], "Country or region"] = "AA"

# Introduce excessively long invalid country names (> 50 characters)
data_copy.loc[length_error_indices[half:], "Country or region"] = "ThisIsAnExtremelyLongAndClearlyInvalidCountryNameThatExceedsFiftyCharacters"

In [372]:
# Error Finding: Identifying invalid length country names
errors = data_copy[
    (data_copy["Country or region"].str.len() < 4) |
    (data_copy["Country or region"].str.len() > 50)
]

print("Number of length errors found:", len(errors))
errors.head(8)

Number of length errors found: 8


,Overall rank,Country or region,Score,GDP per capita,Social support,Healthy life expectancy,Freedom to make life choices,Generosity,Perceptions of corruption
0,1,ThisIsAnExtremelyLongAndClearlyInvalidCountryN...,7.769,1.340,1.587,0.986,0.596,0.153,0.393
2,3,ThisIsAnExtremelyLongAndClearlyInvalidCountryN...,7.554,1.488,1.582,1.028,0.603,0.271,0.341
50,51,ThisIsAnExtremelyLongAndClearlyInvalidCountryN...,6.021,1.500,1.319,0.808,0.493,0.142,0.097
71,72,AA,5.525,1.044,1.303,0.673,0.416,0.133,0.152
84,85,ThisIsAnExtremelyLongAndClearlyInvalidCountryN...,5.265,0.696,1.111,0.245,0.426,0.215,0.041
87,88,AA,5.211,1.002,1.160,0.785,0.086,0.073,0.114
102,103,AA,4.812,0.673,0.799,0.508,0.372,0.105,0.093
138,139,AA,4.085,0.275,0.572,0.410,0.293,0.177,0.085


In [373]:
# Qualitative results + examples

error_indices = errors.index

examples = pd.DataFrame({
    "Original Country": data_original.loc[error_indices, "Country or region"].astype(str),
    "Invalid Country": data_copy.loc[error_indices, "Country or region"].astype(str),
    "Invalid Length": data_copy.loc[error_indices, "Country or region"].str.len().astype(str)
})

examples.head(8)

# After intentionally introducing length errors into approximately 5% of the rows in the **"Country or region"** column, the validity checker successfully detected values whose lengths fall outside the expected range.
# For this test, we assumed that country names should have a reasonable length between **4 and 50 characters**. We introduced two types of length errors: very short values (e.g., "AA") and excessively long values that clearly exceed 50 characters. The checker calculates the string length using `.str.len()` and flags any entry that is shorter than 4 characters or longer than 50 characters.
# Length errors can indicate corrupted text fields, truncated values, or incorrect data entry. Detecting these errors helps preserve the structural integrity of textual columns used for grouping and analysis.



,Original Country,Invalid Country,Invalid Length
0,Finland,ThisIsAnExtremelyLongAndClearlyInvalidCountryN...,75
2,Norway,ThisIsAnExtremelyLongAndClearlyInvalidCountryN...,75
50,Kuwait,ThisIsAnExtremelyLongAndClearlyInvalidCountryN...,75
71,Libya,AA,2
84,Nigeria,ThisIsAnExtremelyLongAndClearlyInvalidCountryN...,75
87,Algeria,AA,2
102,Congo (Brazzaville),AA,2
138,Togo,AA,2


### Look-up Error Check:

In this test, we verify whether values belong to a predefined set of valid entries. A look-up error occurs when a value does not match any value in an approved reference list.

For this dataset, the "Country or region" column should only contain valid country names from the original dataset. Any country name not present in the reference list is considered invalid.

This test will identify rows where the "Country or region" value does not match the approved list of valid country names.

In [374]:
data_copy = data_original.copy()

# Store valid country names from original dataset
valid_countries = set(data_original["Country or region"])

# Select ~5% of rows
lookup_error_indices = sample_indices(data_copy, frac=0.05)

# Introduce invalid country names
fake_values = ["Atlantis", "Middle Earth", "Wakanda", "Narnia"]

for i, idx in enumerate(lookup_error_indices):
    data_copy.loc[idx, "Country or region"] = fake_values[i % len(fake_values)]

In [378]:
# Error Finding: Identifying values not in approved country list
errors = data_copy[~data_copy["Country or region"].isin(valid_countries)]

print("Number of look-up errors found:", len(errors))
errors.head(8)

Number of look-up errors found: 8


,Overall rank,Country or region,Score,GDP per capita,Social support,Healthy life expectancy,Freedom to make life choices,Generosity,Perceptions of corruption
32,33,Narnia,6.293,1.124,1.465,0.891,0.523,0.127,0.150
37,38,Atlantis,6.198,1.246,1.504,0.881,0.334,0.121,0.014
90,91,Middle Earth,5.197,0.987,1.224,0.815,0.216,0.166,0.027
108,109,Wakanda,4.700,0.574,1.122,0.637,0.609,0.232,0.062
124,125,Narnia,4.456,0.562,0.928,0.723,0.527,0.166,0.143
128,129,Atlantis,4.374,0.268,0.841,0.242,0.309,0.252,0.045
137,138,Wakanda,4.107,0.578,1.058,0.426,0.431,0.247,0.087
146,147,Middle Earth,3.597,0.323,0.688,0.449,0.026,0.419,0.110


In [ ]:
# Qualitative results + examples

error_indices = errors.index

examples = pd.DataFrame({
    "Original Country": data_original.loc[error_indices, "Country or region"].astype(str),
    "Invalid Country": data_copy.loc[error_indices, "Country or region"].astype(str)
})

examples.head(8)

# After intentionally introducing look-up errors by replacing approximately 5% of the country names with invalid entries (e.g., "Atlantis", "Wakanda"), the validity checker successfully detected all values not present in the approved reference list.
# The checker compares each value in the "Country or region" column against the set of valid country names extracted from the original dataset. Any value not found in this list is flagged as a look-up error.
# Look-up errors can occur when data is manually entered or merged from inconsistent sources. Detecting them ensures that categorical variables remain aligned with approved reference standards.


,Original Country,Invalid Country
32,Uruguay,Narnia
37,Slovakia,Atlantis
90,Lebanon,Middle Earth
108,Cambodia,Wakanda
124,Bangladesh,Narnia
128,Sierra Leone,Atlantis
137,Zambia,Wakanda
146,Haiti,Middle Earth


## Part 2: Missing Data and Imputation 

### Description
Title: Dataset 2 – Imputation (Insurance / Medical Cost Personal Dataset)

Dataset name: Medical Cost Personal Dataset (Insurance)

Link: https://www.kaggle.com/datasets/mirichoi0218/insurance

Original source: Kaggle dataset (Insurance / Medical Cost Personal Dataset)

Shape: 7 columns, 1338 rows.

Purpose: Contains demographic and lifestyle attributes used to study/predict medical insurance charges (real-world tabular dataset, not synthetic).

Feature description:

- age: age of the primary beneficiary - numerical
- sex: male/female - categorical
- bmi: body mass index - numerical
- children: Number of children covered by health insurance / Number of dependents - numerical
- smoker: yes/no - categorical
- region: the beneficiary's residential area in the US, northeast, southeast, southwest, northwest. - categorical
- charges: medical insurance charges - numerical

In [355]:
##!pip install scikit-learn

In [356]:
# ===============================
# Import Required Libraries
# ===============================

# NumPy: used for numerical operations and random number generation
import numpy as np

# Pandas: used for loading and manipulating tabular datasets
import pandas as pd


# ===============================
# Import Tools for Imputation & Evaluation
# ===============================

# These metrics will be used to quantitatively evaluate
# how close the imputed values are to the original values
from sklearn.metrics import mean_absolute_error, mean_squared_error

# LinearRegression will be used for regression-based imputation (Experiment 2)
from sklearn.linear_model import LinearRegression

# KNNImputer will be used for similarity-based imputation (Experiment 3)
from sklearn.impute import KNNImputer


# ===============================
# Reproducibility Setup
# ===============================

# Setting a fixed random seed ensures that
# the same rows are masked every time the notebook is run.
# This makes results reproducible for the TA.
SEED = 4142

# Create a random number generator using the seed
rng = np.random.default_rng(SEED)


# ===============================
# Evaluation Function
# ===============================

def eval_imputation(y_true, y_pred):
    """
    This function evaluates the quality of imputation.

    Parameters:
    - y_true: the original ground-truth values (before masking)
    - y_pred: the imputed values

    Returns:
    - MAE (Mean Absolute Error)
    - RMSE (Root Mean Squared Error)

    We use these metrics to quantify how far the imputed
    values are from the real values.
    """
    
    # MAE: average absolute difference between true and predicted values
    mae = mean_absolute_error(y_true, y_pred)
    
    # RMSE: square root of average squared differences
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    return {"MAE": mae, "RMSE": rmse}

In [357]:
url = "https://raw.githubusercontent.com/michaelmassaad02/CSI4142_Assignment2/refs/heads/main/insurance.csv"
df = pd.read_csv(url)

print("Shape:", df.shape)
display(df.head())

print("\nMissing values in original dataset:")
display(df.isna().sum())

Shape: (1338, 7)


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520



Missing values in original dataset:


age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

### Experiment 1 — MCAR Missingness on bmi with Median Imputation
In this experiment, missing values will be artificially introduced in the bmi attribute under a Missing Completely At Random (MCAR) mechanism.

MCAR means that the probability of a value being missing is independent of both observed and unobserved data. In this case, 5% of the rows will be randomly selected and their bmi values will be set to missing.

To handle the missing data, we will use median imputation, which replaces missing values with the median of the observed values. The median is chosen because it is robust to outliers and provides a simple baseline imputation method.

The performance of this imputation method will be evaluated quantitatively using MAE and RMSE, comparing the imputed values to the original ground-truth values.

In [358]:
# Create a fresh copy for Experiment 1
df_exp1 = df.copy()

# Save the original ground truth values
bmi_true = df_exp1["bmi"].copy()

# Select 5% of rows completely at random (MCAR)
missing_fraction = 0.05
masked_indices = rng.choice(df_exp1.index, 
                            size=int(len(df_exp1) * missing_fraction), 
                            replace=False)

# Introduce missing values
df_exp1.loc[masked_indices, "bmi"] = np.nan

# Check how many missing values were introduced
print("Number of missing values in bmi after masking:")
print(df_exp1["bmi"].isna().sum())

Number of missing values in bmi after masking:
66


In [359]:
# ===============================
# Detect missing values + Median Imputation
# ===============================

# 1) Detect missing values (before imputation)
missing_before = df_exp1["bmi"].isna().sum()
print("Missing bmi BEFORE imputation:", missing_before)

# 2) Compute median from the OBSERVED (non-missing) bmi values
bmi_median = df_exp1["bmi"].median()
print("Median bmi used for imputation:", bmi_median)

# 3) Impute: replace missing bmi with the median
df_exp1["bmi_imputed"] = df_exp1["bmi"].fillna(bmi_median)

# 4) Verify missing values are resolved in the imputed column
missing_after = df_exp1["bmi_imputed"].isna().sum()
print("Missing bmi AFTER imputation (in bmi_imputed):", missing_after)

# Optional: show a few examples of what changed (masked rows only)
print("\nExamples (masked rows): original bmi vs after imputation")
display(df_exp1.loc[masked_indices[:5], ["bmi", "bmi_imputed"]])

Missing bmi BEFORE imputation: 66
Median bmi used for imputation: 30.38
Missing bmi AFTER imputation (in bmi_imputed): 0

Examples (masked rows): original bmi vs after imputation


,bmi,bmi_imputed
479,NaN,30.38
612,NaN,30.38
314,NaN,30.38
1224,NaN,30.38
498,NaN,30.38


In [360]:
# ===============================
# Quantitative Evaluation (ONLY on masked rows)
# ===============================

# Ground truth values (before masking)
y_true = bmi_true.loc[masked_indices]

# Imputed values (after imputation)
y_pred = df_exp1.loc[masked_indices, "bmi_imputed"]

metrics_exp1 = eval_imputation(y_true, y_pred)
print("Experiment 1 Evaluation Metrics (Median Imputation on BMI):")
print(f"MAE:  {metrics_exp1['MAE']:.3f}")
print(f"RMSE: {metrics_exp1['RMSE']:.3f}")

Experiment 1 Evaluation Metrics (Median Imputation on BMI):
MAE:  5.322
RMSE: 6.734


In this experiment, 5% of the bmi values were randomly removed under a Missing Completely At Random (MCAR) mechanism. Because the missingness was introduced through uniform random sampling, it is independent of both observed and unobserved variables in the dataset.

Median imputation was used to replace the missing values. This method is simple and robust to outliers but does not leverage relationships between variables.

The evaluation results show:

MAE ≈ 5.322

RMSE ≈ 6.734

This indicates that, on average, the imputed BMI values differ from the true values by about 5 units. While median imputation provides a reasonable baseline under MCAR, it does not account for correlations between BMI and other features such as age or charges, which may limit its accuracy.

### Experiment 2 — MAR Missingness on age with Regression Imputation
In this experiment, missing values will be artificially introduced in the age variable under a Missing At Random (MAR) mechanism.

Missing At Random (MAR) means that the probability of a value being missing depends on an observed variable in the dataset. In this case, individuals who are smokers (smoker = "yes") will have a higher probability of having missing age values. Therefore, missingness depends on an observed attribute (smoker) rather than on the value of age itself.

To impute the missing age values, a linear regression model will be trained using complete observations. The model will learn the relationship between age and other predictor variables such as bmi, children, charges, sex, region, and smoker. The trained model will then predict the missing ages.

The imputation quality will be evaluated quantitatively using MAE and RMSE on the artificially masked rows only.

In [361]:
# ===============================
# Create fresh copy for Experiment 2
# ===============================
df_exp2 = df.copy()

# Save original ground truth
age_true = df_exp2["age"].copy()

# 5% missing overall, but dependent on smoker (MAR)
missing_fraction = 0.05

# Get indices where smoker == "yes"
smoker_yes_indices = df_exp2[df_exp2["smoker"] == "yes"].index

# Select 5% of smoker==yes rows
masked_indices_exp2 = rng.choice(
    smoker_yes_indices,
    size=int(len(smoker_yes_indices) * missing_fraction),
    replace=False
)

# Introduce missing values
df_exp2.loc[masked_indices_exp2, "age"] = np.nan

# Check missing values
print("Total missing values in age after masking:")
print(df_exp2["age"].isna().sum())

print("\nMissing values among smokers (yes):")
print(df_exp2.loc[smoker_yes_indices, "age"].isna().sum())

Total missing values in age after masking:
13

Missing values among smokers (yes):
13


In [362]:
# ===============================
# Regression Imputation for age
# ===============================

# One-hot encode categorical variables
df_encoded = pd.get_dummies(df_exp2, drop_first=True)

# Separate rows with and without missing age
train_data = df_encoded[df_encoded["age"].notna()]
test_data = df_encoded[df_encoded["age"].isna()]

# Define predictors (all columns except age)
X_train = train_data.drop(columns=["age"])
y_train = train_data["age"]

X_test = test_data.drop(columns=["age"])

# Train regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Predict missing ages
age_predicted = model.predict(X_test)

# Create a new column for imputed values
df_exp2["age_imputed"] = df_exp2["age"]

# Fill missing age values with predictions
df_exp2.loc[df_exp2["age"].isna(), "age_imputed"] = age_predicted

# Check that missing values are resolved
print("Missing values after regression imputation:")
print(df_exp2["age_imputed"].isna().sum())

Missing values after regression imputation:
0


In [363]:
# ===============================
# Quantitative Evaluation (ONLY on masked rows)
# ===============================

# Ground truth values (before masking)
y_true_exp2 = age_true.loc[masked_indices_exp2]

# Predicted/imputed values
y_pred_exp2 = df_exp2.loc[masked_indices_exp2, "age_imputed"]

metrics_exp2 = eval_imputation(y_true_exp2, y_pred_exp2)

print("Experiment 2 Evaluation Metrics (Regression Imputation on age):")
print(f"MAE:  {metrics_exp2['MAE']:.3f}")
print(f"RMSE: {metrics_exp2['RMSE']:.3f}")

Experiment 2 Evaluation Metrics (Regression Imputation on age):
MAE:  11.957
RMSE: 14.228


In this experiment, missing values in age were introduced under a MAR mechanism, where missingness depended on the observed variable smoker.

Regression imputation was applied using other features as predictors. However, the evaluation results show relatively high MAE and RMSE values (approximately 12–14 years). This suggests that age does not have strong predictive relationships with the other variables in the dataset. As a result, regression imputation does not significantly improve accuracy compared to simpler methods in this case.

This experiment highlights that the effectiveness of model-based imputation depends on the strength of correlations between variables.

### Experiment 3 — MNAR Missingness on charges with KNN (Similarity-Based) Imputation
In this experiment, missing values will be artificially introduced in the charges variable under a Missing Not At Random (MNAR) mechanism.

MNAR means that the probability of a value being missing depends on the value of the variable itself (the missingness is related to the unobserved value). To simulate MNAR, we will make high charges values more likely to be missing by selecting rows in the top 20% of charges and masking a portion of them.

To impute missing charges values, we use a similarity-based method: KNNImputer. KNN imputation fills missing values by finding the most similar rows (nearest neighbors) based on other numeric features and averaging their values.

The imputation quality will be evaluated quantitatively using MAE and RMSE on the artificially masked rows only.

In [364]:
# ===============================
# Create fresh copy for Experiment 3
# ===============================
df_exp3 = df.copy()

# Save original ground truth
charges_true = df_exp3["charges"].copy()

# MNAR setup:
# Top 20% charges group
top_group_threshold = df_exp3["charges"].quantile(0.80)
top_group_idx = df_exp3[df_exp3["charges"] >= top_group_threshold].index

# Mask 25% of the top group => overall missingness ~ 0.20 * 0.25 = 0.05 (5%)
mask_within_top_fraction = 0.25
masked_indices_exp3 = rng.choice(
    top_group_idx,
    size=int(len(top_group_idx) * mask_within_top_fraction),
    replace=False
)

# Introduce missing values in charges
df_exp3.loc[masked_indices_exp3, "charges"] = np.nan

print("Total missing values in charges after masking:", df_exp3["charges"].isna().sum())
print("Missing values within top-charges group:", df_exp3.loc[top_group_idx, "charges"].isna().sum())

Total missing values in charges after masking: 67
Missing values within top-charges group: 67


In [365]:
# ===============================
# KNN Imputation for charges
# ===============================

# Select only numeric columns for KNN
numeric_cols = ["age", "bmi", "children", "charges"]
df_numeric = df_exp3[numeric_cols]

# Initialize KNNImputer
knn_imputer = KNNImputer(n_neighbors=5)

# Apply imputation
df_imputed_array = knn_imputer.fit_transform(df_numeric)

# Convert back to DataFrame
df_imputed = pd.DataFrame(df_imputed_array, columns=numeric_cols)

# Create new column for imputed charges
df_exp3["charges_imputed"] = df_imputed["charges"]

# Check missing values after imputation
print("Missing values after KNN imputation:")
print(df_exp3["charges_imputed"].isna().sum())

Missing values after KNN imputation:
0


In [366]:
# ===============================
# Quantitative Evaluation (ONLY on masked rows)
# ===============================

# Ground truth values (before masking)
y_true_exp3 = charges_true.loc[masked_indices_exp3]

# Imputed values
y_pred_exp3 = df_exp3.loc[masked_indices_exp3, "charges_imputed"]

metrics_exp3 = eval_imputation(y_true_exp3, y_pred_exp3)

print("Experiment 3 Evaluation Metrics (KNN Imputation on charges):")
print(f"MAE:  {metrics_exp3['MAE']:.3f}")
print(f"RMSE: {metrics_exp3['RMSE']:.3f}")

Experiment 3 Evaluation Metrics (KNN Imputation on charges):
MAE:  20019.950
RMSE: 22530.898


In this experiment, missing values were introduced in charges under an MNAR mechanism by masking only high-charge observations. Since missingness depends on the value of the variable itself, this represents the most challenging missing-data scenario.

KNN imputation was applied using similarity between observations based on numeric features. However, the evaluation results show very large MAE and RMSE values (approximately 20,000–22,000). This indicates that KNN struggled to accurately reconstruct extreme charge values.

This experiment demonstrates that imputation methods perform significantly worse under MNAR conditions, especially when extreme values are removed.

### Conclusion — Part 2: Missing Data and Imputation
In this section, three different missing-data mechanisms were simulated and evaluated using distinct imputation methods.

Experiment 1 (MCAR + Median Imputation) showed moderate error. Since missingness was completely random, simple median imputation provided a reasonable baseline. However, it ignored relationships between variables.

Experiment 2 (MAR + Regression Imputation) demonstrated that model-based imputation does not always guarantee improved accuracy. Because age had weak predictive relationships with other features, regression imputation produced relatively high error.

Experiment 3 (MNAR + KNN Imputation) resulted in the largest error values. Since missingness depended on high charges values (extreme cases), similarity-based imputation struggled to accurately reconstruct them. This highlights the difficulty of handling MNAR data.

Overall, the experiments show that imputation performance strongly depends on:

The missing-data mechanism (MCAR vs MAR vs MNAR), and

The strength of relationships between variables.

MNAR scenarios were the most challenging, while simple methods performed reasonably under MCAR conditions.

## Resources:
Choi, M. (2018, February 21). Medical Cost Personal Datasets. Kaggle. https://www.kaggle.com/datasets/mirichoi0218/insurance 

OpenAI. (2024). ChatGPT (GPT-4). https://chat.openai.com/. Chat GPT was used to help determine the validation criterias for the validity checking (Example queries: "How can I validate string formatting in a dataFrame using regular expressions to ensure that a numeric column such as 'Perceptions of corruption' has exactly three digits after the decimal?" "What is a reasonable length range for validating country or region names in a dataset") ChatGPT was also used as a learning support tool to clarify concepts related to missing data mechanisms (MCAR, MAR, MNAR), imputation methods, and evaluation metrics. It also assisted in refining explanations and improving the clarity and structure of written sections.

Sustainable Development Solutions Network. (2019, November 27). World Happiness Report. Kaggle. https://www.kaggle.com/datasets/unsdsn/world-happiness?resource=download&select=2019.csv 